In [15]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings

In [2]:
load_dotenv()

True

In [4]:
groq_key = os.getenv('GROQ_API_KEY')
jina_key = os.getenv('JINA_API_KEY')

print("ENV loaded")


ENV loaded


LOADING OUR DATA

In [5]:
DATA_FILE_PATH = os.path.join("data","hr_policy.txt")

DATA INGESTION

In [8]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")

documents = loader.load()

print("Data loaded")
print("="*40)
print(documents)

Data loaded
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

Langchain Document
Langchain processes everything in form of documents.

DOCUMENTS:
page_content => Actual data 
metadata => extra information about data

In [10]:
print("Total number of characters: ", len(documents[0].page_content))

Total number of characters:  2597


In [12]:
text_splitter = RecursiveCharacterTextSplitter( chunk_size = 500, chunk_overlap = 50)

chunks = text_splitter.split_documents(documents)

print(chunks[0])
print("="*40)

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data\\hr_policy.txt'}


In [14]:
print(chunks[7])
print("="*40)

page_content='7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.' metadata={'source': 'data\\hr_policy.txt'}


In [17]:
embedding_models = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print("Embedding model name: ", embedding_models.model_name)

Embedding model name:  jina-embeddings-v2-base-en


### Store the data in vector DB

In [18]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embedding_models)

print("Chunks are stored", vector_store.index.ntotal)

Chunks are stored 9


In [20]:
test_query = "How many sick leave employee gets ? "

top_matches = vector_store.similarity_search(test_query, k=2)
print(f"Query : {test_query}\n")
for i, match in enumerate(top_matches, start=1):
    print(f"---Match {i}-----")
    print(match.page_content)
    print()

Query : How many sick leave employee gets ? 

---Match 1-----
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

---Match 2-----
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



### Data retrieval

In [21]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.3
)

llm.model_name

'openai/gpt-oss-120b'

In [23]:
test_response = llm.invoke("What is conformal prediction ? Answer in 4 line.")

print(test_response.content)

Conformal prediction is a statistical framework that wraps any machine‑learning model with a mathematically‑rigorous guarantee on its error rate.  
It produces prediction sets (or intervals) that contain the true outcome with a user‑specified confidence level (e.g., 95%).  
The method works by assessing how “non‑conforming” a new example is compared to a calibration set of past data.  
Thus, it yields valid, distribution‑free uncertainty quantification for classification, regression, and beyond.


### AI Agents

In [33]:
# Tools

retriever = vector_store.as_retriever(search_kwargs={"k":3})

def search_hr_policy(question: str) -> str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

In [34]:
from langchain.agents import create_agent


hr_assistant = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt=""" 
    You are HR Policy Assistant. Always use the search_hr_policy tool to lookup facts before answering. 
    If the answer isn't in the search results, say you don't know instead of guessing."
    """
)

print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [35]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [36]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which holidays."
            }
        ]
    }
)

In [40]:
print(response["messages"][-1].content)

The HR policy states that the company observes **12 public holidays each year**. These holidays are taken from the official holiday calendar that HR publishes at the start of every calendar year. If you need the specific dates or names of those holidays for the current year, you can refer to that calendar on the HR portal.


In [41]:
print(ask_hr_assistant("How many sick leave do we get ? "))

QUESTION: How many sick leave do we get ? 
------------------------------------------------------------
ANSWER: According to the company’s leave policy, each full‑time employee is entitled to **10 paid sick days per year**. (A medical certificate is required if the sick leave exceeds two consecutive days.)

According to the company’s leave policy, each full‑time employee is entitled to **10 paid sick days per year**. (A medical certificate is required if the sick leave exceeds two consecutive days.)
